# 🗂️ Notebook 2: Shopping Cart — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right corner of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab only needs `pydantic` — no Redis, no database. We simulate everything in plain Python so you can focus on the *ideas*.


## The mental model

A cart is a **slim, per-user collection of product references**. Nothing more.

Entities:

| Entity | Lives in | Notes |
|---|---|---|
| `Product` | Product Service | name, price, stock |
| `CartItem` | Cart DB | `{sku, qty}` — that's it |
| `Cart` | Cart DB | belongs to one user or one guest session |
| `Reservation` | Inventory Svc | short-lived lock at checkout (see NB3) |
| `Order` | Order Svc | immutable, after payment |

The Cart DB **does not** store prices, names, images. Keeping it slim is the main design rule.

## Bad vs. Best: modelling the cart

### 🚫 Bad: a plain `dict`, no validation

Lots of tutorials start here. It's easy to type but every bug is a runtime surprise.

In [ ]:
# BAD: no types, no validation, nothing stops garbage input.
bad_cart = {"user": "alice", "items": [{"sku": "A", "qty": -3}]}  # negative qty!
bad_cart["items"].append({"sku": "B"})                             # qty missing!
print(bad_cart)
# Nothing errored. Bug will explode somewhere in checkout.

### ✅ Best: Pydantic models with invariants

Pydantic lets us declare the shape *and* the constraints. Bad data is rejected at the boundary.

In [ ]:
from decimal import Decimal
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    sku: str
    name: str
    price: Decimal          # always use Decimal for money, never float
    stock: int = Field(ge=0)

class CartItem(BaseModel):
    sku: str
    qty: int = Field(ge=1, le=10)   # anti-hoarding cap (source: max 10 per item)

class Cart(BaseModel):
    user_id: str            # or guest_session_id — both are strings
    items: list[CartItem] = Field(default_factory=list, max_length=50)  # anti-hoarding cap

    def total(self, catalog: dict[str, Product]) -> Decimal:
        # Prices come from the catalog, NOT from the cart itself.
        return sum((catalog[i.sku].price * i.qty for i in self.items), Decimal(0))

catalog = {
    "A": Product(sku="A", name="Python Crash Course", price=Decimal("29.99"), stock=100),
    "B": Product(sku="B", name="Clean Code",          price=Decimal("35.00"), stock=50),
}
cart = Cart(user_id="u1", items=[CartItem(sku="A", qty=2), CartItem(sku="B", qty=1)])
print("cart total:", cart.total(catalog))

# Bad data is now rejected up front:
try:
    CartItem(sku="X", qty=-1)
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])

### Why are there size caps?

`max_length=50` on `items` and `le=10` on `qty` are not arbitrary. During flash sales, bots try to **hoard inventory** by adding 10 000 of something to their cart (soft stock checks will still let you add it). Caps stop this at the API boundary. Real Amazon has similar per-item limits.

## Primary key design (NoSQL)

If you're on DynamoDB / Cassandra / Bigtable:

```
Partition Key = user_id   (or guest_session_id)
Sort Key      = sku       (so each item is its own row)
```

Why two keys instead of blobbing the whole cart into one JSON value?

- **Updating one item** (change qty from 2 → 3) rewrites just that row, not the whole cart.
- **Fetching the whole cart** is still one partition read.
- **Deleting an item** is one row delete.

A single-row-per-cart design (blob) is simpler but makes concurrent updates
painful: two devices both read the blob, both modify their copy, and the second
write silently erases the first. That is the **lost update**, and it is the
single most famous problem in shopping-cart design — Amazon's Dynamo paper is
largely about it. Notebook 3 reproduces it, then fixes it three different ways.

Note the per-item design is not free either: a `GET /cart` is now a partition
*query* rather than a single-item read, and "empty the cart" becomes N deletes
instead of one. You have traded write conflicts for read cost.

## HTTP API

| Method | Path | Auth | Purpose |
|---|---|---|---|
| `GET`    | `/cart`                       | session or user | Current cart (hydrated with prices) |
| `POST`   | `/cart/items`                 | session or user | Add item |
| `PATCH`  | `/cart/items/{sku}`           | session or user | Change quantity |
| `DELETE` | `/cart/items/{sku}`           | session or user | Remove item |
| `POST`   | `/cart/merge`                 | user only       | Merge guest cart into user cart |
| `POST`   | `/checkout` (`Idempotency-Key`) | user only     | Reserve + charge (NB 3) |

Notice three things:

1. **Guests and logged-in users share the same endpoints.** The gateway resolves `session_token` → temporary `user_id = "guest_<sessionid>"`.
2. **`/cart/merge` exists.** This is a first-class flow, not a side-effect of login.
3. **`/checkout` takes an Idempotency-Key.** Money endpoints always do.

## Idempotency — the right way

**Problem:** user taps *Pay* → network blip → client retries → are they charged twice?

**Solution:** client generates a fresh UUID per checkout attempt and sends it in the `Idempotency-Key` header. Server remembers the *response* keyed by that UUID. A retry with the same key returns the cached response instead of re-running the work.

The naive demo below runs the "happy replay" case. The tricky case is **network failure** *before* the server finishes — the client doesn't know if it succeeded, so it retries. The server must handle that too.

In [ ]:
import time

# Idempotency store: key -> (state, response, started_at)
store: dict[str, tuple[str, dict | None, float]] = {}
LOCK_TTL = 30  # seconds

def charge(key: str, amount: int) -> dict:
    now = time.time()
    entry = store.get(key)

    if entry:
        state, resp, started = entry
        if state == "done":
            print(f"  replay '{key}' → cached response")
            return resp
        if state == "in_flight" and now - started < LOCK_TTL:
            # Concurrent request with same key — tell client to wait or retry later.
            print(f"  '{key}' still in flight — 409 Conflict")
            return {"status": "in_progress"}

    # Mark in flight BEFORE doing the work so concurrent retries see it.
    store[key] = ("in_flight", None, now)
    # ... call payment gateway ...
    resp = {"order_id": f"ord_{key[:4]}", "amount": amount, "status": "paid"}
    store[key] = ("done", resp, now)
    print(f"  '{key}' charged → {resp['order_id']}")
    return resp

# Scenario: first call succeeds, client retries twice (maybe after a timeout).
print(charge("abc-123", 50))
print(charge("abc-123", 50))   # exact same response, no double charge
print(charge("xyz-999", 50))   # new key → new order

## Guest → User merge: API contract

This is the trickiest cart flow. A user:
1. Browses anonymously, gets `guest_session_id = "gs_abc"`.
2. Adds 2× Book-A, 1× Book-B to their guest cart.
3. Logs in. Their saved cart has 1× Book-A, 1× Book-C.
4. What should `/cart/merge` do?

**Source says:** for overlapping SKUs, **sum quantities** (capped). For non-overlapping, add to the user cart. Then **delete the guest cart** so a retry doesn't double up.

### Idempotent by design

The request body includes a **merge_id** (UUID). If the server sees the same merge_id twice, it's a replay — return the already-merged cart. This is the same pattern as `/checkout`.

```json
POST /cart/merge
{
  "guest_session_id": "gs_abc",
  "merge_id": "6f90f5e4-..."
}
```

We'll implement the merge algorithm in Notebook 3.

## TTLs — keep the store clean

| Data | TTL | Why |
|---|---|---|
| Cart cache entry (Redis) | 24 h of inactivity | RAM is precious |
| Guest cart (DB) | 7–30 days | Anonymous carts pile up |
| User cart (DB) | 30 days or more | People really do come back |
| Product metadata cache | ~5 min | Prices change |

NoSQL stores (DynamoDB, Cassandra) have native TTL — set an `expires_at`
attribute and the store evicts rows for free.

Three things the TTL table above does not tell you, and Notebook 3 demonstrates:

- **Native TTL is best-effort.** DynamoDB documents deletion "typically within
  48 hours" of expiry. If your abandoned-cart email job reads the table
  directly it will see carts that are logically dead. Always filter on
  `expires_at` at read time as well; treat TTL as garbage collection, not as
  a correctness mechanism.
- **Every write must push the TTL forward**, or an active shopper's cart
  vanishes mid-session because `expires_at` was set on the *first* write.
- **Abandonment is a product event, not a deletion.** "Cart untouched for 1
  hour" is when marketing wants an email; "cart untouched for 30 days" is when
  storage wants the row gone. Two different clocks — do not conflate them.

## Takeaways

- Cart DB stores **only** `{user_id, sku, qty}`. Prices live elsewhere.
- **Pydantic caps** (qty ≤ 10, ≤ 50 items) are an anti-hoarding firewall.
- NoSQL key design: PK = `user_id`, SK = `sku`.
- **Idempotency-Key** is non-negotiable on money endpoints — and useful for `/cart/merge` too.
- The **merge** flow is a first-class API, not a login side-effect.

Next: Notebook 3 turns these ideas into runnable Python.